# GPT-2 (`sdadas/polish-gpt2-medium`) + LoRA na PolEmo2

Wyniki zapisywane są na Dysku Google (`OUT`, `AUG`). Po rozłączeniu środowiska zakończone przebiegi
(z `results.json`) i gotowa augmentacja są pomijane, a przerwany trening wznawia się od checkpointu ostatniej epoki.

| Część | Wynik |
|---|---|
| klasyfikacja tekstu (LoRA, head) | `OUT/gpt2-text`, `OUT/gpt2-text-head` |
| klasyfikacja tokenów | `OUT/gpt2-token` |
| embeddingi ostatniego tokenu | `OUT/gpt2-text/plots` |
| augmentacja generatywna | `AUG/augmented_gpt2_*.json` |
| trening na augmentacji + tabela | `OUT/gpt2-text-augmented`, `OUT/gpt2_comparison.md` |

In [1]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/nlp_lab/outputs"
AUG = "/content/drive/MyDrive/nlp_lab/augmented"

Mounted at /content/drive


In [2]:
%cd /content
!git clone -b gpt2 https://github.com/mtorzynska/nlp_language_models_lab.git || git -C nlp_language_models_lab pull
%cd /content/nlp_language_models_lab
# bez -U: aktualizacja moze podmienic torch z cuda na wersje bez cuda
!pip install -q "transformers>=5" "datasets>=4" peft accelerate umap-learn
# stara preinstalowana torchao psuje import peft, lora jej nie potrzebuje
!pip uninstall -y -q torchao
!python -m src.data.prepare

import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK GPU")

/content
Cloning into 'nlp_language_models_lab'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 157 (delta 72), reused 133 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 1.93 MiB | 2.31 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/nlp_language_models_lab
<frozen runpy>:130: RuntimeWarning: 'src.data.prepare' found in sys.modules after import of package 'src.data', but prior to execution of 'src.data.prepare'; this may result in unpredictable behaviour

all_text/train/0000.parquet: downloading bytes:  29% 1.04M/3.63M [00:00<00:02, 1.12MB/s]
all_text/train/0000.parquet: downloading bytes: 100% 3.61M/3.61M [00:01<00:00, 3.35MB/s,  351kB/s  ]
all_text/train/0000.parquet: reconstructing file: 100% 3.63M/3.63M [00:01<00:00, 3.36MB/s,  353kB/s  ]
0000.parquet: 100% 447k/447k [00:00<00:00, 20.9MB/s]
0000.parquet: 100% 440k/440k [00:00<00:00, 118MB/s]
G

In [3]:
from pathlib import Path

for run in ["gpt2-text", "gpt2-text-head", "gpt2-token", "gpt2-text-augmented"]:
    path = Path(OUT) / run
    checkpoints = sorted((path / "checkpoints").glob("checkpoint-*"), key=lambda c: int(c.name.split("-")[1]))
    state = "gotowe" if (path / "results.json").exists() else f"przerwane ({checkpoints[-1].name})" if checkpoints else "brak"
    print(f"{run:22} {state}")
print(f"{'augmentacja':22} {'gotowe' if Path(AUG, 'augmented_gpt2_examples.md').exists() else 'brak'}")

gpt2-text              gotowe
gpt2-text-head         gotowe
gpt2-token             gotowe
gpt2-text-augmented    gotowe
augmentacja            gotowe


## Klasyfikacja tekstu

Szybki test na małym podzbiorze i pełny trening. Batch 4 × akumulacja 4 = efektywnie 16.

In [4]:
!python -m src.training.gpt2 --task text --max-train-samples 16 --max-eval-samples 8 --epochs 1 --max-length 64 --output-dir /content/test_run

config.json: 100% 837/837 [00:00<00:00, 4.07MB/s]
tokenizer_config.json: 100% 321/321 [00:00<00:00, 1.97MB/s]
special_tokens_map.json: 100% 72.0/72.0 [00:00<00:00, 418kB/s]
tokenizer.json: 100% 2.34M/2.34M [00:00<00:00, 68.9MB/s]
Generating train split: 6573 examples [00:00, 201011.73 examples/s]
Generating validation split: 823 examples [00:00, 169194.79 examples/s]
Generating test split: 820 examples [00:00, 195150.32 examples/s]
Map: 100% 16/16 [00:00<00:00, 859.47 examples/s]
Map: 100% 8/8 [00:00<00:00, 920.16 examples/s]
Map: 100% 8/8 [00:00<00:00, 836.60 examples/s]

model.safetensors: downloading bytes:  12% 180M/1.53G [00:01<00:06, 195MB/s, 16.4MB/s  ]
model.safetensors: downloading bytes:  14% 210M/1.53G [00:01<00:07, 178MB/s, 18.4MB/s  ]
model.safetensors: downloading bytes:  56% 849M/1.53G [00:03<00:02, 290MB/s, 67.4MB/s  ]
model.safetensors: downloading bytes:  62% 952M/1.53G [00:05<00:05, 113MB/s, 66.4MB/s  ] 
model.safetensors: downloading bytes:  63% 957M/1.53G [00:15<00

In [5]:
!python -m src.training.gpt2 --task text --batch-size 4 --grad-accum 4 --output-dir {OUT}/gpt2-text
!cat {OUT}/gpt2-text/test_report.txt

/content/drive/MyDrive/nlp_lab/outputs/gpt2-text ma juz results.json, pomijam (--overwrite trenuje od nowa)
              precision    recall  f1-score   support

        zero     0.9831    0.9831    0.9831       118
       minus     0.9147    0.9174    0.9161       339
        plus     0.9110    0.9471    0.9287       227
         amb     0.7381    0.6838    0.7099       136

    accuracy                         0.8963       820
   macro avg     0.8867    0.8829    0.8844       820
weighted avg     0.8942    0.8963    0.8950       820


Baseline do porównania: zamrożony GPT-2, trenowana tylko głowica (bez LoRA). Większy learning rate, bo uczy się tylko jedna warstwa.

In [6]:
!python -m src.training.gpt2 --task text --mode head --lr 1e-3 --batch-size 4 --grad-accum 4 --output-dir {OUT}/gpt2-text-head
!cat {OUT}/gpt2-text-head/test_report.txt

/content/drive/MyDrive/nlp_lab/outputs/gpt2-text-head ma juz results.json, pomijam (--overwrite trenuje od nowa)
              precision    recall  f1-score   support

        zero     0.9658    0.9576    0.9617       118
       minus     0.8669    0.9027    0.8844       339
        plus     0.8577    0.9295    0.8922       227
         amb     0.6442    0.4926    0.5583       136

    accuracy                         0.8500       820
   macro avg     0.8337    0.8206    0.8242       820
weighted avg     0.8416    0.8500    0.8436       820


## Klasyfikacja tokenów

Każde słowo dostaje sentyment swojego zdania. Etykieta tylko na pierwszym subtokenie słowa, `<eos>` i padding mają `-100`.

In [7]:
!python -m src.training.gpt2 --task token --batch-size 4 --grad-accum 4 --output-dir {OUT}/gpt2-token
!cat {OUT}/gpt2-token/test_report.txt

/content/drive/MyDrive/nlp_lab/outputs/gpt2-token ma juz results.json, pomijam (--overwrite trenuje od nowa)
              precision    recall  f1-score   support

        zero     0.8698    0.8518    0.8607     20725
       minus     0.8454    0.8802    0.8624     28795
        plus     0.7491    0.8241    0.7848     14061
         amb     0.5668    0.3998    0.4689      7334

    accuracy                         0.8111     70915
   macro avg     0.7578    0.7390    0.7442     70915
weighted avg     0.8046    0.8111    0.8058     70915


## Embeddingi ostatniego tokenu

Reprezentacja ostatniego prawdziwego tokenu (wejście głowicy klasyfikacyjnej) dla recenzji testowych:
bazowy GPT-2 (LoRA wyłączona) i po strojeniu. Wykresy UMAP i t-SNE, puste znaczniki to błędne predykcje.

In [8]:
if not Path(f"{OUT}/gpt2-text/plots/gpt2_embeddings.npz").exists():
    !python -m src.evaluation.gpt2_embeddings --adapter {OUT}/gpt2-text/adapter

In [9]:
from pathlib import Path
from IPython.display import HTML, display

for name in ["gpt2_base_umap", "gpt2_finetuned_umap"]:
    display(HTML(Path(f"{OUT}/gpt2-text/plots/{name}.html").read_text(encoding="utf-8")))

In [14]:
import numpy as np
from sklearn.metrics import silhouette_score
d = np.load(f"{OUT}/gpt2-text/plots/gpt2_embeddings.npz")
for v in ["base", "finetuned"]:
    print(v, round(silhouette_score(d[f"{v}_embeddings"], d["labels"], metric="cosine"), 4))
print("accuracy", (d["predictions"] == d["labels"]).mean())

base 0.0489
finetuned 0.3076
accuracy 0.8963414634146342


## Augmentacja generatywna

Początek recenzji treningowej (połowa słów, maks. 40) jest promptem, GPT-2 dopisuje resztę, a nowy tekst
dostaje etykietę recenzji źródłowej. Dwa warianty próbkowania:
`conservative` (temperature 0.7, top-k 50, top-p 0.9) i `diverse` (temperature 1.0, bez top-k, top-p 0.95).

In [10]:
if not Path(f"{AUG}/augmented_gpt2_examples.md").exists():
    !python -m src.data.gpt2_augmentation --max-samples 2000 --output-dir {AUG}

In [11]:
from IPython.display import Markdown

display(Markdown(Path(f"{AUG}/augmented_gpt2_examples.md").read_text(encoding="utf-8")))

| wariant | etykieta | oryginal | po augmentacji (prompt + generacja) |
|---|---|---|---|
| conservative | minus | Chciał em zestaw do podstawowych prac coś przykręcić coś odkręcić - jako , ze nie jestem wielkim majsterkowiczem nie wymagał em wiele jednak sam zestaw jest bardzo mały - wielkości połowy kartki A4 zarówno grzechotka jaki i wkrętak - gubią się w dłoni . Najgorsze jest spasowanie elementów nasadek … | Chciał em zestaw do podstawowych prac coś przykręcić coś odkręcić - jako , ze nie jestem wielkim majsterkowiczem nie wymagał em wiele jednak sam zestaw jest bardzo mały - wielkości połowy kartki A4 zarówno grzechotka jaki i wkrętak - gubią się w nim wszystkie śruby, śrubki itp - nie ma możliwości … |
| conservative | minus | własciwie nic się nie dowidział em , podczas badania usłyszał em , , wszystko mięciutko " i że jest cos zaczerwienione , ale co to jest to niewiadomo , recepta na czopki i jakby się cos działo to , , proszę przyjsć " , akurat za 150 zł na wizytę która razem z badaniem trwała tyle mniej więcej ile … | własciwie nic się nie dowidział em , podczas badania usłyszał em , , wszystko mięciutko " i że jest cos zaczerwienione , ale co to jest to niewiadomo , recepta na czopki i jakby się cos działo to , , że trzeba to zgłosić i jak nie to , że jak nic nie będzie to , albo , że , że to trzeba zrobić. … |
| conservative | zero | Gdy Dominika Nowis zastanawia się , jak zniszczyć raka - zawsze sprawdza , czy jej metody walki nie zniszczą naszych zdrowych organów . To ona jako pierwsza w świecie zbadała , czy inhibitory proteasomu upośledzają funkcję serca . Prace , jakie prowadziła razem z dr . Gaetano Vattemim z … | Gdy Dominika Nowis zastanawia się , jak zniszczyć raka - zawsze sprawdza , czy jej metody walki nie zniszczą naszych zdrowych organów . To ona jako pierwsza w świecie zbadała , czy inhibitory proteasomu upośledzają funkcję serca . Prace , które trwały ponad 100 lat, dowiodły , że inhibitor nie … |
| diverse | minus | Chciał em zestaw do podstawowych prac coś przykręcić coś odkręcić - jako , ze nie jestem wielkim majsterkowiczem nie wymagał em wiele jednak sam zestaw jest bardzo mały - wielkości połowy kartki A4 zarówno grzechotka jaki i wkrętak - gubią się w dłoni . Najgorsze jest spasowanie elementów nasadek … | Chciał em zestaw do podstawowych prac coś przykręcić coś odkręcić - jako , ze nie jestem wielkim majsterkowiczem nie wymagał em wiele jednak sam zestaw jest bardzo mały - wielkości połowy kartki A4 zarówno grzechotka jaki i wkrętak - gubią się młotki (które świetnie się poluzują i dopiero na … |
| diverse | minus | własciwie nic się nie dowidział em , podczas badania usłyszał em , , wszystko mięciutko " i że jest cos zaczerwienione , ale co to jest to niewiadomo , recepta na czopki i jakby się cos działo to , , proszę przyjsć " , akurat za 150 zł na wizytę która razem z badaniem trwała tyle mniej więcej ile … | własciwie nic się nie dowidział em , podczas badania usłyszał em , , wszystko mięciutko " i że jest cos zaczerwienione , ale co to jest to niewiadomo , recepta na czopki i jakby się cos działo to , , że trzeba pisać to : dieta , leki , wizyty . Potem mu udało się dojechać do domu w którym ja … |
| diverse | zero | Gdy Dominika Nowis zastanawia się , jak zniszczyć raka - zawsze sprawdza , czy jej metody walki nie zniszczą naszych zdrowych organów . To ona jako pierwsza w świecie zbadała , czy inhibitory proteasomu upośledzają funkcję serca . Prace , jakie prowadziła razem z dr . Gaetano Vattemim z … | Gdy Dominika Nowis zastanawia się , jak zniszczyć raka - zawsze sprawdza , czy jej metody walki nie zniszczą naszych zdrowych organów . To ona jako pierwsza w świecie zbadała , czy inhibitory proteasomu upośledzają funkcję serca . Prace , na które powołuje się Dominika Nowisa, nie są jedynie … |


## Trening na danych augmentowanych

Tak jak w części HerBERTa: train to same wygenerowane przykłady, walidacja i test bez zmian.
Pliki `augmented_mask_*_train.json` z augmentacji HerBERTa obecne w `AUG` trafiają do tego samego treningu
(jeden model na wszystkich augmentacjach).

In [12]:
import glob

files = [f"{AUG}/augmented_gpt2_combined_train.json"] + sorted(glob.glob(f"{AUG}/augmented_mask_*_train.json"))
print(files)
AUG_FILES = " ".join(files)
!python -m src.training.gpt2 --task text --batch-size 4 --grad-accum 4 --augmented {AUG_FILES} --output-dir {OUT}/gpt2-text-augmented
!cat {OUT}/gpt2-text-augmented/test_report.txt

['/content/drive/MyDrive/nlp_lab/augmented/augmented_gpt2_combined_train.json']
/content/drive/MyDrive/nlp_lab/outputs/gpt2-text-augmented ma juz results.json, pomijam (--overwrite trenuje od nowa)
              precision    recall  f1-score   support

        zero     0.9825    0.9492    0.9655       118
       minus     0.8659    0.8761    0.8710       339
        plus     0.8536    0.8987    0.8755       227
         amb     0.6048    0.5515    0.5769       136

    accuracy                         0.8390       820
   macro avg     0.8267    0.8189    0.8222       820
weighted avg     0.8360    0.8390    0.8371       820


In [13]:
!python -m src.evaluation.gpt2_comparison --root {OUT}
display(Markdown(Path(f"{OUT}/gpt2_comparison.md").read_text(encoding="utf-8")))

| przebieg | zadanie | tryb | augmentacja | train | accuracy | macro F1 | Δ macro F1 | F1 zero | F1 minus | F1 plus | F1 amb |
|---|---|---|---|---|---|---|---|---|---|---|---|
| gpt2-text | tekst | lora | nie | 6573 | 0.8963 | 0.8844 | — | 0.9831 | 0.9161 | 0.9287 | 0.7099 |
| gpt2-text-augmented | tekst | lora | tak | 3910 | 0.8390 | 0.8222 | -0.0622 | 0.9655 | 0.8710 | 0.8755 | 0.5769 |
| gpt2-text-head | tekst | head | nie | 6573 | 0.8500 | 0.8242 | — | 0.9617 | 0.8844 | 0.8922 | 0.5583 |
| gpt2-token | tokeny | lora | nie | 4451 | 0.8111 | 0.7442 | — | 0.8607 | 0.8624 | 0.7848 | 0.4689 |



| przebieg | zadanie | tryb | augmentacja | train | accuracy | macro F1 | Δ macro F1 | F1 zero | F1 minus | F1 plus | F1 amb |
|---|---|---|---|---|---|---|---|---|---|---|---|
| gpt2-text | tekst | lora | nie | 6573 | 0.8963 | 0.8844 | — | 0.9831 | 0.9161 | 0.9287 | 0.7099 |
| gpt2-text-augmented | tekst | lora | tak | 3910 | 0.8390 | 0.8222 | -0.0622 | 0.9655 | 0.8710 | 0.8755 | 0.5769 |
| gpt2-text-head | tekst | head | nie | 6573 | 0.8500 | 0.8242 | — | 0.9617 | 0.8844 | 0.8922 | 0.5583 |
| gpt2-token | tokeny | lora | nie | 4451 | 0.8111 | 0.7442 | — | 0.8607 | 0.8624 | 0.7848 | 0.4689 |
